In [ ]:
# Colab dependency install (T4 / CUDA + QLoRA)
!pip install -U bitsandbytes transformers peft datasets evaluate bert-score rouge_score huggingface_hub trl accelerate tensorboard

In [ ]:
# Imports
import math
import random
import re
from functools import partial
from typing import Any

import evaluate
import numpy as np
import torch
from datasets import concatenate_datasets, load_dataset
from google.colab import userdata
# from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
from peft import LoraConfig, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    EarlyStoppingCallback,
    TrainerCallback,
    pipeline,
)
from trl import SFTConfig, SFTTrainer

In [ ]:
# Authentication + runtime checks
hf_token = userdata.get("HF_TOKEN_WRITE") # For Google Colab
# hf_token = UserSecretsClient().get_secret("HF_TOKEN") # For Kaggle Notebook
if hf_token:
    login(token=hf_token)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA runtime not detected. Set Colab runtime to GPU (T4).")

print(f"CUDA device: {torch.cuda.get_device_name(0)}")

CUDA device: Tesla T4


In [ ]:
# Configuration
SEED = 42  # Global random seed for reproducibility across dataset mapping and training.
MODEL_ID = "google/medgemma-1.5-4b-it"  # Base Hugging Face model ID to fine-tune.
PROMPT = """
System Prompt:
You are a medical scribe. Given a transcription of a doctor-patient consultation, generate a structured clinical note in the following format. Write in third person, past tense, using formal medical language. Synthesize and paraphrase — do not copy dialogue verbatim.
Required sections and guidance:
CHIEF COMPLAINT — One sentence stating the reason for the visit.
HISTORY OF PRESENT ILLNESS — Narrative paragraph(s) covering: patient demographics and relevant past medical history (PMH), then chronological account of current status for each active condition, lifestyle factors, medication adherence, and any symptoms endorsed or denied. Group related conditions together.
REVIEW OF SYSTEMS — Bulleted by organ system. Use "Endorses" or "Denies" for each relevant symptom. Only include systems discussed in the encounter.
PHYSICAL EXAMINATION — List findings by system. Only include what was explicitly examined.
VITALS REVIEWED — Summarize notable vitals only (e.g., "Elevated," "Within normal limits"). Omit if not discussed.
RESULTS — Summarize any lab, imaging, or diagnostic results referenced during the visit. Omit if none discussed.
ASSESSMENT AND PLAN — Restate patient demographics and PMH in the opening sentence. Then for each active diagnosis (in order of clinical priority), include relevant subsections chosen from: Medical Reasoning, Additional Testing, Medical Treatment, Patient Education and Counseling. Close with "Patient Agreements" confirming the patient's understanding.
Rules:

Only document what is present in the transcription; do not invent clinical details.
Use standard medical abbreviations and terminology (e.g., dyspnea, ideation, ejection fraction).
If a section has no relevant content from the transcript, omit it entirely.
Keep the tone objective and professional throughout.

Input: A raw transcription of a doctor-patient conversation.
Output: A complete clinical note following the format above.

Input:
{}

Output:
"""  # Single fixed instruction prompt used for all training examples.

TRAIN_NOISE_PROB = 0.02  # Probability of dropping each word in train dialogues to simulate ASR errors.
VALID_NOISE_PROB = 0.0  # Noise level for validation data; 0 keeps clean validation targets.
TEST_NOISE_PROB = 0.0  # Noise level for test-time prompts during final evaluation.

MAX_SEQ_LENGTH = 4096  # Maximum token length for prompt + target sequence truncation.
MAX_NEW_TOKENS = 2048  # Maximum tokens generated per inference sample during evaluation.

NUM_TRAIN_EPOCHS = 32  # Upper bound on epochs; early stopping may stop sooner.
LEARNING_RATE = 1e-5  # Initial learning rate for LoRA adapter optimization.
PER_DEVICE_TRAIN_BATCH_SIZE = 1  # Micro-batch size on T4 GPU.
PER_DEVICE_EVAL_BATCH_SIZE = 1  # Evaluation micro-batch size.
GRADIENT_ACCUMULATION_STEPS = 4  # Accumulation steps to emulate larger effective batch size.
WARMUP_STEPS = 10  # Number of steps used for LR warmup.
MAX_GRAD_NORM = 0.3  # Gradient clipping norm threshold for training stability.

EVAL_STEPS = 5  # Evaluate every N optimizer steps.
SAVE_STEPS = 5  # Save checkpoint every N optimizer steps.
LOGGING_STEPS = 5  # Emit training logs every N steps.
EARLY_STOPPING_PATIENCE = 5  # Number of evaluations without improvement before stopping.

EVAL_SUBSET_SIZE = 128  # Validation subset size used during training for faster iteration.
TEST_SUBSET_SIZE = 256  # Test subset size for final metric reporting in Colab-friendly runtime.

OUTPUT_DIR = "outputs/medgemma-aci-lora-t4"  # Local directory for checkpoints and adapter artifacts.
PUSH_TO_HUB = True  # If True, upload adapter checkpoints to Hugging Face Hub.
HUB_MODEL_ID = "gabrielbuzzi/medgemma-aci-lora"  # Hub repo ID used only when PUSH_TO_HUB=True.

REPORT_TO = "tensorboard"  # Tracking backend for training metrics visualization.

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# Dataset load + split selection
dataset = load_dataset("mkieffer/ACI-Bench", "aci")
train_ds = dataset["train"]
valid_ds = dataset["valid"]

print(train_ds)
print(valid_ds)

Dataset({
    features: ['encounter_id', 'dialogue', 'note'],
    num_rows: 35
})
Dataset({
    features: ['encounter_id', 'dialogue', 'note'],
    num_rows: 11
})


In [ ]:
# Preprocessing helpers

def clean_dialogue(text: str) -> str:
    """Remove speaker tags/newlines to better match raw transcription style."""
    text = re.sub(r"\[doctor\]", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\[patient\]", "", text, flags=re.IGNORECASE)
    text = text.replace("\\n", " ").replace("\n", " ")
    return re.sub(r"\s+", " ", text).strip()


def simulate_transcription_noise(text: str, drop_prob: float) -> str:
    """Drop random words to approximate ASR omissions."""
    words = text.split()
    words = [w for w in words if random.random() > drop_prob]
    return " ".join(words)


def format_train_valid_example(prompt: str, example: dict[str, Any], noise_prob: float) -> dict[str, Any]:
    dialogue = clean_dialogue(example["dialogue"])
    if noise_prob > 0:
        dialogue = simulate_transcription_noise(dialogue, drop_prob=noise_prob)

    example["messages"] = [
        {
            "role": "user",
            "content": [{"type": "text", "text": prompt.format(dialogue)}],
        },
        {
            "role": "assistant",
            "content": [{"type": "text", "text": example["note"]}],
        },
    ]
    return example


def format_test_example(prompt: str, example: dict[str, Any], noise_prob: float) -> dict[str, Any]:
    dialogue = clean_dialogue(example["dialogue"])
    if noise_prob > 0:
        dialogue = simulate_transcription_noise(dialogue, drop_prob=noise_prob)

    example["messages"] = [
        {
            "role": "user",
            "content": [{"type": "text", "text": prompt.format(dialogue)}],
        }
    ]
    return example

train_ds = train_ds.map(partial(format_train_valid_example, PROMPT, noise_prob=TRAIN_NOISE_PROB))
valid_ds = valid_ds.map(partial(format_train_valid_example, PROMPT, noise_prob=VALID_NOISE_PROB))

print(train_ds[0]["messages"])

[{'content': [{'text': '\nSystem Prompt:\nYou are a medical scribe. Given a transcription of a doctor-patient consultation, generate a structured clinical note in the following format. Write in third person, past tense, using formal medical language. Synthesize and paraphrase — do not copy dialogue verbatim.\nRequired sections and guidance:\nCHIEF COMPLAINT — One sentence stating the reason for the visit.\nHISTORY OF PRESENT ILLNESS — Narrative paragraph(s) covering: patient demographics and relevant past medical history (PMH), then chronological account of current status for each active condition, lifestyle factors, medication adherence, and any symptoms endorsed or denied. Group related conditions together.\nREVIEW OF SYSTEMS — Bulleted by organ system. Use "Endorses" or "Denies" for each relevant symptom. Only include systems discussed in the encounter.\nPHYSICAL EXAMINATION — List findings by system. Only include what was explicitly examined.\nVITALS REVIEWED — Summarize notable vi

In [ ]:
# Tokenization/collator helpers
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token


def collate_fn(examples: list[dict[str, Any]]) -> dict[str, torch.Tensor]:
    texts = []
    token_type_ids_list = []
    for example in examples:
        user_text = " ".join(
            " ".join(part["text"] for part in msg["content"])
            for msg in example["messages"]
            if msg["role"] == "user"
        )
        assistant_text = " ".join(
            " ".join(part["text"] for part in msg["content"])
            for msg in example["messages"]
            if msg["role"] == "assistant"
        )
        full_text = f"{user_text}{tokenizer.eos_token}{assistant_text}{tokenizer.eos_token}"
        texts.append(full_text)
        user_ids = tokenizer(user_text + tokenizer.eos_token, add_special_tokens=False)["input_ids"]
        assistant_ids = tokenizer(assistant_text + tokenizer.eos_token, add_special_tokens=False)["input_ids"]
        token_type_ids_list.append([0] * len(user_ids) + [1] * len(assistant_ids))

    batch = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        add_special_tokens=False,
    )
    labels = batch["input_ids"].clone()
    labels[labels == tokenizer.pad_token_id] = -100
    max_len = batch["input_ids"].size(1)
    padded_token_type_ids = []
    for ids in token_type_ids_list:
        if len(ids) < max_len:
            ids = ids + [0] * (max_len - len(ids))
        else:
            ids = ids[:max_len]
        padded_token_type_ids.append(ids)
    batch["token_type_ids"] = torch.tensor(padded_token_type_ids, dtype=torch.long)
    batch["labels"] = labels
    return batch


def preprocess_logits_for_metrics(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    if isinstance(logits, tuple):
        logits = logits[0]
    return torch.argmax(logits, dim=-1)


def compute_token_metrics(eval_preds: tuple[np.ndarray, np.ndarray]) -> dict[str, float]:
    pred_ids, label_ids = eval_preds

    pred_ids = np.asarray(pred_ids)
    label_ids = np.asarray(label_ids)

    shifted_preds = pred_ids[:, :-1]
    shifted_labels = label_ids[:, 1:]

    valid_mask = shifted_labels != -100
    valid_count = int(valid_mask.sum())
    if valid_count == 0:
        return {"token_accuracy": 0.0}

    correct = (shifted_preds == shifted_labels) & valid_mask
    token_accuracy = float(correct.sum() / valid_count)
    return {"token_accuracy": token_accuracy}


class EvalPerplexityCallback(TrainerCallback):
    """Adds eval_perplexity into eval metrics based on eval_loss."""

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is not None and "eval_loss" in metrics:
            eval_loss = metrics["eval_loss"]
            if eval_loss is not None and eval_loss < 20:
                metrics["eval_perplexity"] = float(math.exp(eval_loss))

config.json:   0%|          | 0.00/2.55k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

In [ ]:
# Model load (T4-specific QLoRA)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, # FP16 as using T4
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    dtype=torch.bfloat16, # FP16 as using T4 (using dtype because torch_dtype returns ``torch_dtype` is deprecated! Use `dtype` instead!`)
    device_map="auto",
)

for name, param in model.named_parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)

model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id
model.config.bos_token_id = tokenizer.bos_token_id
if hasattr(model, "generation_config"):
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    model.generation_config.eos_token_id = tokenizer.eos_token_id
    model.generation_config.bos_token_id = tokenizer.bos_token_id

model.gradient_checkpointing_enable()
model.config.use_cache = False

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

In [ ]:
# LoRA config + attach adapters
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 19,248,896 || all params: 4,319,328,368 || trainable%: 0.4456


In [ ]:
# Trainer args + callbacks
eval_subset_size = min(EVAL_SUBSET_SIZE, len(valid_ds))
eval_ds_for_training = valid_ds.shuffle(seed=SEED).select(range(eval_subset_size))

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    hub_model_id=HUB_MODEL_ID,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    max_grad_norm=MAX_GRAD_NORM,
    lr_scheduler_type="linear",
    fp16=False, # True for T4
    bf16=True, # True for A100, H100, RTX 30-series, RTX 40-series
    eval_strategy="steps",
    save_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_steps=SAVE_STEPS,
    logging_steps=LOGGING_STEPS,
    logging_strategy="steps",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    push_to_hub=PUSH_TO_HUB,
    report_to=REPORT_TO,
    seed=SEED,
    dataset_kwargs={"skip_prepare_dataset": True},
    remove_unused_columns=False,
    label_names=["labels"],
    dataset_text_field=None,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds_for_training,
    processing_class=tokenizer,
    data_collator=collate_fn,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE),
        EvalPerplexityCallback(),
    ],
    compute_metrics=compute_token_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
)

In [ ]:
%tensorboard --logdir {OUTPUT_DIR}

In [ ]:
# Train + best-checkpoint restore
train_result = trainer.train()

print(train_result)
print(trainer.state.best_model_checkpoint)

Step,Training Loss,Validation Loss,Token Accuracy,Perplexity
5,2.184648,2.100894,0.527383,8.173475
10,2.203915,2.039996,0.539544,7.690578
15,2.101813,1.963083,0.553731,7.121247
20,2.003778,1.887411,0.570392,6.602253
25,1.959530,1.807167,0.589444,6.093159
30,1.856133,1.738198,0.607240,5.687084
35,1.792656,1.684146,0.618955,5.387846
40,1.771610,1.635588,0.633427,5.132474
45,1.717942,1.590411,0.642791,4.905766
50,1.672697,1.550912,0.649276,4.715767


TrainOutput(global_step=288, training_loss=1.5296949197848637, metrics={'train_runtime': 972.3575, 'train_samples_per_second': 1.152, 'train_steps_per_second': 0.296, 'total_flos': 5.422714705649664e+16, 'train_loss': 1.5296949197848637})
outputs/medgemma-aci-lora-t4/checkpoint-275


In [ ]:
model_id = "google/medgemma-1.5-4b-it" # or the specific base model you used
adapter_path = "./outputs/medgemma-aci-lora-t4/checkpoint-275"

# 2. Push the adapter directly
# This only uploads the LoRA weights, not the full base model
from peft import PeftConfig, PeftModel

model = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(model_id),
    adapter_path
)

model.push_to_hub("gabrielbuzzi/medgemma-aci-lora", token=hf_token)

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 17.0kB / 77.1MB            

CommitInfo(commit_url='https://huggingface.co/gabrielbuzzi/medgemma-aci-lora/commit/3475b4cf772edf0ccaccae404d44a0e859759e81', commit_message='Upload model', commit_description='', oid='3475b4cf772edf0ccaccae404d44a0e859759e81', pr_url=None, repo_url=RepoUrl('https://huggingface.co/gabrielbuzzi/medgemma-aci-lora', endpoint='https://huggingface.co', repo_type='model', repo_id='gabrielbuzzi/medgemma-aci-lora'), pr_revision=None, pr_num=None)

In [ ]:
# Save/push adapter
trainer.save_model()
if PUSH_TO_HUB:
    trainer.push_to_hub(token=hf_token)

HfHubHTTPError: (Request ID: Root=1-699ccd37-2720a33a00708ac429cb7510;ff0d1769-dca6-4c85-a382-e3ab85ed4ce3)

403 Forbidden: You don't have the rights to create a model under the namespace "your-username".
Cannot access content at: https://huggingface.co/api/repos/create.
Make sure your token has the correct permissions.

In [ ]:
# Evaluation pipeline + final metrics report
rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")
bertscore_metric = evaluate.load("bertscore")


def generate_predictions(eval_dataset, text_pipe, tokenizer_obj, max_new_tokens: int) -> list[str]:
    predictions = []
    for example in eval_dataset:
        # Keep only system + user messages, drop the assistant turn
        inference_messages = [m for m in example["messages"] if m["role"] != "assistant"]

        prompt_text = tokenizer_obj.apply_chat_template(
            inference_messages,
            tokenize=False,
            add_generation_prompt=True,  # adds the assistant header to prompt generation
        )
        output = text_pipe(
            prompt_text,
            max_new_tokens=max_new_tokens,  # uncomment this too
            return_full_text=False,
        )
        print(output)
        predictions.append(output[0]["generated_text"].strip())
    return predictions


def compute_cross_entropy_and_perplexity(references: list[str], eval_model, eval_tokenizer) -> tuple[float, float]:
    eval_model.eval()
    total_loss = 0.0
    total_tokens = 0

    with torch.no_grad():
        for ref in references:
            inputs = eval_tokenizer(ref, return_tensors="pt", truncation=True, padding=True).to(eval_model.device)
            outputs = eval_model(**inputs, labels=inputs["input_ids"])
            loss = outputs.loss
            token_count = int(inputs["input_ids"].numel())
            total_loss += float(loss.item() * token_count)
            total_tokens += token_count

    avg_ce = total_loss / max(total_tokens, 1)
    ppl = float(math.exp(avg_ce)) if avg_ce < 20 else float("inf")
    return avg_ce, ppl


def compute_text_metrics(predictions: list[str], references: list[str], eval_model, eval_tokenizer) -> dict[str, float]:
    rouge = rouge_metric.compute(predictions=predictions, references=references)
    bleu = bleu_metric.compute(predictions=predictions, references=[[r] for r in references])
    bertscore = bertscore_metric.compute(predictions=predictions, references=references, lang="en")
    ce, ppl = compute_cross_entropy_and_perplexity(references, eval_model, eval_tokenizer)

    return {
        "rouge1": float(rouge["rouge1"]),
        "rouge2": float(rouge["rouge2"]),
        "rougeL": float(rouge["rougeL"]),
        "bleu": float(bleu["bleu"]),
        "bertscore_f1": float(np.mean(bertscore["f1"])),
        "cross_entropy": ce,
        "perplexity": ppl,
    }


test_ds = concatenate_datasets([dataset["test1"], dataset["test2"], dataset["test3"]])
test_ds = test_ds.map(partial(format_test_example, PROMPT, noise_prob=TEST_NOISE_PROB))

if TEST_SUBSET_SIZE > 0:
    test_ds = test_ds.select(range(min(TEST_SUBSET_SIZE, len(test_ds))))

inference_model = trainer.model
inference_model.eval()

text_pipe = pipeline(
    "text-generation",
    model=inference_model,
    tokenizer=tokenizer,
    device_map="auto",
    do_sample=False,
    max_new_tokens=MAX_NEW_TOKENS,  # set it here as the default
    pad_token_id=tokenizer.eos_token_id,
    temperature=0.0,
)

references = test_ds["note"]
predictions = generate_predictions(test_ds, text_pipe, tokenizer, MAX_NEW_TOKENS)
final_metrics = compute_text_metrics(predictions, references, text_pipe.model, tokenizer)

print("\n===== Final Evaluation Metrics (Fine-Tuned Model) =====")
for metric_name, metric_value in final_metrics.items():
    print(f"{metric_name}: {metric_value:.4f}")

Map:   0%|          | 0/66 [00:00<?, ? examples/s]

NameError: name 'trainer' is not defined

## Evaluation

In [ ]:
import torch
import math
import numpy as np
import evaluate

from transformers import (
    AutoProcessor,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline,
)
from peft import PeftModel
from datasets import concatenate_datasets
from functools import partial
from tqdm import tqdm


BASE_MODEL_ID = "google/medgemma-1.5-4b-it"
CHECKPOINT_PATH = "gabrielbuzzi/medgemma-aci-lora"

MAX_NEW_TOKENS = 1024


print("Loading processor...")
processor = AutoProcessor.from_pretrained(BASE_MODEL_ID)
tokenizer = processor.tokenizer
tokenizer.padding_side = "left"


print("Loading BASE model (no quantization)...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

base_model.eval()


print("Loading LoRA on top of BASE...")
fine_tuned_model = PeftModel.from_pretrained(
    base_model,
    CHECKPOINT_PATH,
)

print("Merging LoRA weights...")
fine_tuned_model = fine_tuned_model.merge_and_unload()
fine_tuned_model.eval()


print("Building inference pipelines...")

base_pipe = pipeline(
    "text-generation",
    model=base_model,
    tokenizer=tokenizer,
)

ft_pipe = pipeline(
    "text-generation",
    model=fine_tuned_model,
    tokenizer=tokenizer,
)


for pipe in [base_pipe, ft_pipe]:
    pipe.model.generation_config.do_sample = False
    pipe.model.generation_config.temperature = 0.0
    pipe.model.generation_config.pad_token_id = tokenizer.eos_token_id



Loading processor...


The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
`torch_dtype` is deprecated! Use `dtype` instead!


Loading BASE model (no quantization)...


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Loading LoRA on top of BASE...
Merging LoRA weights...
Building inference pipelines...


In [ ]:
from tqdm import tqdm

rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")
bertscore_metric = evaluate.load("bertscore")


def generate_predictions(eval_dataset, pipe, processor, max_new_tokens):
    predictions = []

    for example in tqdm(eval_dataset):
        inference_messages = [
            m for m in example["messages"] if m["role"] != "assistant"
        ]

        prompt_text = processor.tokenizer.apply_chat_template(
            inference_messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        output = pipe(
            prompt_text,
            max_new_tokens=max_new_tokens,
            return_full_text=False,
        )

        predictions.append(output[0]["generated_text"].strip())

    return predictions


def compute_cross_entropy_and_perplexity(references, model, tokenizer):
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    with torch.no_grad():
        for ref in references:
            inputs = tokenizer(
                ref,
                return_tensors="pt",
                truncation=True,
                padding=True,
            ).to(model.device)

            outputs = model(**inputs, labels=inputs["input_ids"])
            loss = outputs.loss

            token_count = int(inputs["input_ids"].numel())
            total_loss += float(loss.item() * token_count)
            total_tokens += token_count

    avg_ce = total_loss / max(total_tokens, 1)
    ppl = float(math.exp(avg_ce)) if avg_ce < 20 else float("inf")

    return avg_ce, ppl



def compute_text_metrics(predictions, references, model, tokenizer):
    print("Computing ROUGE...")
    rouge = rouge_metric.compute(predictions=predictions, references=references)

    print("Computing BLEU...")
    bleu = bleu_metric.compute(
        predictions=predictions,
        references=[[r] for r in references],
    )

    print("Computing BERTScore...")
    bertscore = bertscore_metric.compute(
        predictions=predictions,
        references=references,
        lang="en",
    )

    print("Computing Cross-Entropy + Perplexity...")
    ce, ppl = compute_cross_entropy_and_perplexity(
        references,
        model,
        tokenizer,
    )

    return {
        "rouge1": float(rouge["rouge1"]),
        "rouge2": float(rouge["rouge2"]),
        "rougeL": float(rouge["rougeL"]),
        "bleu": float(bleu["bleu"]),
        "bertscore_f1": float(np.mean(bertscore["f1"])),
        "cross_entropy": ce,
        "perplexity": ppl,
    }

In [ ]:
test_ds = concatenate_datasets([
    dataset["test1"]
])

test_ds = test_ds.map(
    partial(format_test_example, PROMPT, noise_prob=TEST_NOISE_PROB)
)

if TEST_SUBSET_SIZE > 0:
    test_ds = test_ds.select(range(min(TEST_SUBSET_SIZE, len(test_ds))))

references = test_ds["note"]
MAX_NEW_TOKENS = 278

print("\nGenerating FINE-TUNED predictions...")
ft_predictions = generate_predictions(test_ds, ft_pipe, processor, MAX_NEW_TOKENS)

# print("\nGenerating BASE predictions...")
# base_predictions = generate_predictions(test_ds, base_pipe, processor, MAX_NEW_TOKENS)

# print("\nComputing BASE metrics...")
# base_metrics = compute_text_metrics(base_predictions, references, base_model, tokenizer)

print("\nComputing FINE-TUNED metrics...")
ft_metrics = compute_text_metrics(ft_predictions, references, fine_tuned_model, tokenizer)

print("\n================= FINAL COMPARISON =================")

for key in base_metrics.keys():
    print(f"{key}")
    #print(f"   Base Model      : {base_metrics[key]:.4f}")
    print(f"   Fine-Tuned Model: {ft_metrics[key]:.4f}")
    print("--------------------------------------------------")


Generating FINE-TUNED predictions...


 32%|███▏      | 7/22 [03:44<07:37, 30.50s/it]You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=278) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
100%|██████████| 22/22 [11:14<00:00, 30.64s/it]



Computing FINE-TUNED metrics...
Computing ROUGE...
Computing BLEU...
Computing BERTScore...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Computing Cross-Entropy + Perplexity...

================= FINAL COMPARISON =================


NameError: name 'base_metrics' is not defined

In [ ]:
ft_metrics

{'rouge1': 0.37056410486975866,
 'rouge2': 0.12364718704464037,
 'rougeL': 0.18522573329025183,
 'bleu': 0.04934890886005067,
 'bertscore_f1': 0.8388870412653143,
 'cross_entropy': 2.463799742498255,
 'perplexity': 11.749371414234243}

In [ ]:
ft_predictions[0]

'<unused94>thought\nThe user wants me to create a clinical note based on the provided doctor-patient conversation. I need to follow a specific format and use standard medical language.\n\n**Input Analysis:**\nThe conversation is between a doctor (Dr. Unknown) and a patient named Charles.\n- **Chief Complaint:** Charles reports shortness of breath, specifically mid-shortness of breath, and believes it\'s related to allergies, possibly exacerbated by his cat. He has a history of allergies, uses an inhaler, and has seen an urgent care for it.\n- **History of Present Illness (HPI):**\n    - Patient demographics: Charles, male, moved to the city 2 years ago, owns land, is a "huge hunter," has a refrigerator full of venison.\n    - PMH: Denies allergies in the past.\n    - Current Status: Noticed breathing is "n\'t what it used to be," catches breath more often.\n    - Symptoms: Mid-shortness of breath (sudden onset), possibly triggered by cat/outside.\n    - Associated Symptoms: Eyebugers (

In [ ]:
print("\nComputing FINE-TUNED metrics...")
ft_metrics = compute_text_metrics(ft_predictions, references, fine_tuned_model, tokenizer)


Computing FINE-TUNED metrics...
Computing ROUGE...
Computing BLEU...
Computing BERTScore...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Computing Cross-Entropy + Perplexity...


In [ ]:
ft_metrics

{'rouge1': 0.4200228153707498,
 'rouge2': 0.13276701709730612,
 'rougeL': 0.18441733817403014,
 'bleu': 0.05545459458839209,
 'bertscore_f1': 0.7997348687865518,
 'cross_entropy': 2.311936624634165,
 'perplexity': 10.093953938317354}

In [ ]:
print("\nGenerating BASE predictions...")
base_predictions = generate_predictions(test_ds, base_pipe, processor, MAX_NEW_TOKENS)

print("\nComputing BASE metrics...")
base_metrics = compute_text_metrics(base_predictions, references, base_model, tokenizer)

print("\nComputing FINE-TUNED metrics...")
ft_metrics = compute_text_metrics(ft_predictions, references, fine_tuned_model, tokenizer)

print("\n================= FINAL COMPARISON =================")

for key in base_metrics.keys():
    print(f"{key}")
    print(f"   Base Model      : {base_metrics[key]:.4f}")
    print(f"   Fine-Tuned Model: {ft_metrics[key]:.4f}")
    print("--------------------------------------------------")


Generating BASE predictions...


 91%|█████████ | 20/22 [09:48<00:58, 29.09s/it]Both `max_new_tokens` (=278) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
base_predictions[0]

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model_id = "google/medgemma-1.5-4b-it" # Use the exact base model you tuned
checkpoint_path = "./outputs/medgemma-aci-lora-t4/checkpoint-275"
export_path = "./merged_medgemma_model"

# 1. Load the Base Model
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.bfloat16, # Recommended for Gemma
    device_map="cpu",           # Merge on CPU to save VRAM if needed
)

# 2. Load the Adapter onto the Base Model
model = PeftModel.from_pretrained(base_model, checkpoint_path)

# 3. Merge the weights
# This effectively performs the weight addition: W_merged = W_base + (A * B)
merged_model = model.merge_and_unload()

# 4. Load and save the tokenizer (crucial for a complete model)
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

# 5. Push everything to Hugging Face
# This will upload the full 5GB+ model, not just the tiny adapter
merged_model.push_to_hub("gabrielbuzzi/medgemma-aci-lora-merged")
tokenizer.push_to_hub("gabrielbuzzi/medgemma-aci-lora-merged")